# LiveTrainer Hot-Reloading Demo

This notebook demonstrates the **LiveTrainer** feature for Kubeflow Trainer v2.

LiveTrainer enables **real-time hot-reloading of hyperparameters** (learning rate, momentum, weight decay, etc.) during long-running distributed training jobs — without stopping or restarting training.

## How It Works

1. Training pods and a Kubeflow Notebook mount the **same shared volume** (any RWX storage: NFS, CephFS, PVC, etc.)
2. A **polling-based FileWatcher** monitors a YAML config file on the shared volume for changes
3. At configurable step intervals, **Rank 0 checks for changes** and broadcasts updates to all workers via `torch.distributed.broadcast`
4. Updates are **applied directly to optimizer param groups** without process restart

```
┌──────────────┐     writes      ┌──────────────┐     reads       ┌──────────────┐
│   Notebook   │ ──────────────► │Shared Volume │ ◄────────────── │ Training Pod │
│  (User edits │    params.yaml  │  /mnt/shared │   FileWatcher   │  (Rank 0)    │
│   params)    │                 │              │   polls mtime    │              │
└──────────────┘                 └──────────────┘                  └──────┬───────┘
                                                                         │
                                                              dist.broadcast()
                                                                         │
                                                                  ┌──────▼───────┐
                                                                  │ Training Pod │
                                                                  │  (Rank 1..N) │
                                                                  └──────────────┘
```

## 1. Setup and Imports

In [ ]:
from kubeflow.trainer import LiveTrainer, SyncConfig

## 2. Verify LiveTrainer and SyncConfig

In [ ]:
# SyncConfig defaults
sync = SyncConfig()
print(f"sync_interval:       {sync.sync_interval}")
print(f"cooldown_seconds:    {sync.cooldown_seconds}")
print(f"config_file_name:    {sync.config_file_name}")
print(f"use_lock_file:       {sync.use_lock_file}")
print(f"checksum_validation: {sync.checksum_validation}")

In [ ]:
# SyncConfig with custom values
sync_custom = SyncConfig(
    sync_interval=20,
    cooldown_seconds=1.0,
    config_file_name="hyperparams.yaml",
    checksum_validation=True,
)
print(f"Custom sync config: interval={sync_custom.sync_interval}, "
      f"cooldown={sync_custom.cooldown_seconds}s, "
      f"file={sync_custom.config_file_name}")

## 3. Validation Tests

The dataclasses validate inputs on construction.

In [ ]:
# Test: sync_interval must be positive
try:
    SyncConfig(sync_interval=0)
except ValueError as e:
    print(f"[PASS] SyncConfig(sync_interval=0): {e}")

# Test: cooldown must be non-negative
try:
    SyncConfig(cooldown_seconds=-1)
except ValueError as e:
    print(f"[PASS] SyncConfig(cooldown_seconds=-1): {e}")

# Test: func must be callable
try:
    LiveTrainer(func="not a function")
except ValueError as e:
    print(f"[PASS] LiveTrainer(func='not a function'): {e}")

# Test: shared_volume_mount_path must be non-empty
try:
    LiveTrainer(func=lambda: None, shared_volume_mount_path="")
except ValueError as e:
    print(f"[PASS] LiveTrainer(shared_volume_mount_path=''): {e}")

## 4. Define the Training Function

The training function uses `_kubeflow_sync_loop` and `_apply_param_updates` — these globals are injected by the LiveTrainer instrumentation at runtime.

In [ ]:
def train_fn():
    import torch
    import torch.nn as nn

    # Simple model and optimizer
    model = nn.Linear(10, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    loss_fn = nn.MSELoss()

    print(f"[train] Initial lr={optimizer.param_groups[0]['lr']}, "
          f"momentum={optimizer.param_groups[0]['momentum']}")

    for epoch in range(5):
        for step in range(100):
            # Check for parameter updates at the configured interval
            if _kubeflow_sync_loop.should_sync():  # noqa: F821
                updates = _kubeflow_sync_loop.sync_params()  # noqa: F821
                if updates:
                    _apply_param_updates(optimizer, updates)  # noqa: F821
                    print(f"[train] epoch={epoch} step={step}: Applied updates: {updates}")

            # Normal training step
            x = torch.randn(32, 10)
            y = torch.randn(32, 1)
            pred = model(x)
            loss = loss_fn(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"[train] epoch={epoch} done, lr={optimizer.param_groups[0]['lr']:.6f}, "
              f"loss={loss.item():.4f}")

    print("[train] Training complete.")

## 5. Create the LiveTrainer

The shared volume (NFS, CephFS, or any RWX PVC) must be provisioned and mounted separately — for example, via `PodTemplateOverrides` or cluster-level configuration.

In [ ]:
trainer = LiveTrainer(
    func=train_fn,
    shared_volume_mount_path="/mnt/shared",  # Where the RWX volume is mounted
    hot_reload_params=["learning_rate", "momentum", "weight_decay"],
    sync_config=SyncConfig(
        sync_interval=10,            # Check every 10 steps
        cooldown_seconds=0.5,        # Wait 0.5s after file change
        checksum_validation=True,    # MD5 validation against partial writes
    ),
    num_nodes=2,
    resources_per_node={"gpu": 1},
)

print("LiveTrainer created:")
print(f"  func:                    {trainer.func.__name__}")
print(f"  shared_volume_mount_path: {trainer.shared_volume_mount_path}")
print(f"  hot_reload_params:       {trainer.hot_reload_params}")
print(f"  num_nodes:               {trainer.num_nodes}")

In [ ]:
from kubeflow.trainer.livetrainer.runtime import get_live_trainer_instrumentation_wrapper

wrapper = get_live_trainer_instrumentation_wrapper(trainer)
# Show the first part of the generated wrapper
lines = wrapper.split("\n")
print(f"Generated wrapper: {len(lines)} lines total")
print("---")
# Print the initialization section (skip the function definition body)
in_init_section = False
for line in lines:
    if "Initialize instrumentation" in line:
        in_init_section = True
    if in_init_section:
        print(line)
    if "USER TRAINING CODE" in line:
        break

In [ ]:
# Verify the wrapper compiles
test_code = wrapper.replace("{{user_func_import_and_call}}", "pass")
compile(test_code, "<test>", "exec")
print("[PASS] Generated wrapper code compiles successfully.")

## 7. Test FileWatcher and SyncControlLoop Locally

We can test the runtime instrumentation directly (without Kubernetes) by calling `_create_live_trainer_instrumentation`.

In [ ]:
import os
import tempfile
import time

import yaml

from kubeflow.trainer.livetrainer.runtime import _create_live_trainer_instrumentation

# Create a temporary directory to simulate a shared volume
tmpdir = tempfile.mkdtemp(prefix="livetrainer_demo_")
config_path = os.path.join(tmpdir, "params.yaml")
lock_path = os.path.join(tmpdir, "params.yaml.lock")

print(f"Simulated shared volume directory: {tmpdir}")
print(f"Config file: {config_path}")

In [ ]:
# Initialize the instrumentation (single-process mode — no torch.distributed)
sync_loop, apply_fn = _create_live_trainer_instrumentation(
    config_file_path=config_path,
    sync_interval=5,
    cooldown_seconds=0,
    use_lock_file=False,
    lock_file_path=lock_path,
    checksum_validation=True,
    hot_reload_params=["learning_rate", "momentum"],
)

print("SyncControlLoop and apply_param_updates created.")

In [ ]:
# Test 1: No config file yet — sync returns None
result = sync_loop.sync_params()
print(f"Test 1 - No config file: sync_params() = {result}")
assert result is None, "Expected None when no config file exists"
print("[PASS]")

In [ ]:
# Test 2: Write initial config — sync detects it
with open(config_path, "w") as f:
    yaml.dump({"learning_rate": 0.001, "momentum": 0.95, "secret": "filtered"}, f)

result = sync_loop.sync_params()
print(f"Test 2 - Initial config: sync_params() = {result}")
assert result == {"learning_rate": 0.001, "momentum": 0.95}, "Expected filtered params"
assert "secret" not in result, "'secret' should be filtered out by hot_reload_params whitelist"
print("[PASS] Correctly filtered by hot_reload_params whitelist")

In [ ]:
# Test 3: No change — sync returns None
result = sync_loop.sync_params()
print(f"Test 3 - No change: sync_params() = {result}")
assert result is None, "Expected None when file hasn't changed"
print("[PASS]")

In [ ]:
# Test 4: Modify config — sync detects the change
time.sleep(0.05)  # ensure mtime differs
with open(config_path, "w") as f:
    yaml.dump({"learning_rate": 0.0005, "momentum": 0.99}, f)

result = sync_loop.sync_params()
print(f"Test 4 - Modified config: sync_params() = {result}")
assert result == {"learning_rate": 0.0005, "momentum": 0.99}
print("[PASS]")

In [ ]:
# Test 5: should_sync() respects interval
sync_loop2, _ = _create_live_trainer_instrumentation(
    config_file_path=config_path,
    sync_interval=3,
    cooldown_seconds=0,
    use_lock_file=False,
    lock_file_path=lock_path,
    checksum_validation=False,
    hot_reload_params=[],
)

results = [sync_loop2.should_sync() for _ in range(9)]
print(f"Test 5 - should_sync() over 9 steps: {results}")
assert results == [False, False, True, False, False, True, False, False, True]
print("[PASS] Syncs every 3 steps")

In [ ]:
# Test 6: Invalid YAML is handled gracefully
time.sleep(0.05)
with open(config_path, "w") as f:
    f.write("invalid: yaml: [")

sync_loop3, _ = _create_live_trainer_instrumentation(
    config_file_path=config_path,
    sync_interval=1,
    cooldown_seconds=0,
    use_lock_file=False,
    lock_file_path=lock_path,
    checksum_validation=False,
    hot_reload_params=[],
)

result = sync_loop3.sync_params()
print(f"Test 6 - Invalid YAML: sync_params() = {result}")
assert result is None, "Expected None on YAML parse error"
print("[PASS] Invalid YAML handled gracefully")

In [ ]:
# Cleanup
import shutil

shutil.rmtree(tmpdir)
print(f"Cleaned up {tmpdir}")

## 8. Test CRD Generation

Verify that LiveTrainer generates a valid Trainer CRD with the instrumented command.

In [ ]:
from kubeflow.trainer.constants import constants
from kubeflow.trainer.livetrainer.utils import get_trainer_cr_from_live_trainer
from kubeflow.trainer.types.types import Runtime, RuntimeTrainer, TrainerType

# Create a mock runtime
runtime_trainer = RuntimeTrainer(
    trainer_type=TrainerType.CUSTOM_TRAINER,
    framework="pytorch",
    image="pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime",
)
runtime_trainer.set_command(constants.TORCH_COMMAND)
runtime = Runtime(name="torch-distributed", trainer=runtime_trainer)

# Build the Trainer CRD
trainer_cr = get_trainer_cr_from_live_trainer(runtime, trainer)

print("Trainer CRD:")
print(f"  num_nodes: {trainer_cr.num_nodes}")
print(f"  image:     {trainer_cr.image}")
print(f"  command:   {len(trainer_cr.command)} parts")
print(f"  env:       {trainer_cr.env}")

# Verify key instrumentation markers in the command
cmd = " ".join(trainer_cr.command)
assert "_kubeflow_sync_loop" in cmd, "Missing _kubeflow_sync_loop"
assert "_apply_param_updates" in cmd, "Missing _apply_param_updates"
assert "train_fn" in cmd, "Missing user function"
assert "FileWatcher" in cmd, "Missing FileWatcher class"
assert "SyncControlLoop" in cmd, "Missing SyncControlLoop class"
print("\n[PASS] Trainer CRD contains all required instrumentation.")

## 9. Submit to Kubernetes (Live Cluster)

**Uncomment the cells below** to submit the LiveTrainer to an actual Kubernetes cluster with Kubeflow Trainer v2 installed.

### Prerequisites
- Kubeflow Trainer v2 operator running in the cluster
- A shared RWX volume accessible from training pods
- `kubectl` configured with appropriate context

In [ ]:
# # Submit the training job
# client = TrainerClient()
# job_name = client.train(trainer=trainer)
# print(f"TrainJob created: {job_name}")

In [ ]:
# # Watch the job status
# trainjob = client.wait_for_job_status(
#     name=job_name,
#     status={"Running"},
#     timeout=300,
# )
# print(f"TrainJob {job_name} is {trainjob.status}")

In [ ]:
# # Hot-reload: update learning rate while training is running!
# # Write a new params.yaml to the shared volume (from this notebook)
# import yaml
#
# shared_mount = "/mnt/shared"  # Path where the shared volume is mounted in this notebook
# with open(f"{shared_mount}/params.yaml", "w") as f:
#     yaml.dump({"learning_rate": 0.0001, "momentum": 0.99}, f)
# print("Updated params.yaml — training pods will pick up changes at next sync interval.")

In [ ]:
# # Stream logs to see the parameter update applied
# for line in client.get_job_logs(name=job_name, follow=True):
#     print(line)
#     if "Training complete" in line:
#         break

## Summary

| Feature | Description |
|---------|-------------|
| **FileWatcher** | Polls config file via mtime + optional MD5 checksum (no inotify needed) |
| **SyncControlLoop** | Step-based sync with `should_sync()` + `sync_params()` |
| **Distributed Broadcast** | Rank 0 reads file, broadcasts via `torch.distributed.broadcast` |
| **apply_param_updates** | Maps `learning_rate` -> `lr` and applies to optimizer param groups |
| **Param Whitelist** | `hot_reload_params` filters which params are allowed |
| **Error Handling** | YAML errors, missing files, and partial writes handled gracefully |
| **Shared Volume** | User provisions RWX volume (NFS, CephFS, PVC, etc.) separately |